In [1]:
import colorsys
import math
import os
from pathlib import Path

import click
import h3pandas
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydeck as pdk
import tensorflow as tf
import yaml

from lib.models.encoders import CoordEncoder
from lib.models.tf_gp_model import TFGeoPriorModel
from sinr_paper_evals import load_class_to_taxa_inat_dataset, load_class_to_taxa_sinr_dataset

2025-01-26 16:41:44.014796: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-01-26 16:41:44.014824: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-26 16:41:44.015763: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-26 16:41:44.020128: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-01-26 16:41:44.584649: W tensorflow/compiler/tf2

In [2]:
config_file = "config_inat_2_17_coords.yml"
with open(config_file, "r") as f:
    config = yaml.safe_load(f)

In [14]:
def coord_grid(grid_size):
    feats = np.zeros(
        (grid_size[0], grid_size[1], 2),
        dtype=np.float32
    )
    mg = np.meshgrid(
        np.linspace(-180, 180, feats.shape[1]),
        np.linspace(90, -90, feats.shape[0]),
    )
    feats[:, :, 0] = mg[0]
    feats[:, :, 1] = mg[1]

    return feats.reshape(feats.shape[0] * feats.shape[1], 2)
    

In [15]:
if config["input_type"] == "coords+env":
    raster = np.load(config["bioclim_data"]).astype(np.float32)
elif config["input_type"] == "coords_elev":
    raster = np.load(config["elev_data"]).astype(np.float32)
else:
    raster = None
encoder = CoordEncoder(config["input_type"], raster)
model = TFGeoPriorModel(config["model_save_name"])

In [16]:
print("loading class to taxa")
if config["dataset_type"] == "sinr":
    class_to_taxa = load_class_to_taxa_sinr_dataset(
        config["sinr_dataset"]["train_data"]
    )
elif config["dataset_type"] == "inat":
    class_to_taxa = load_class_to_taxa_inat_dataset(
        config["inat_dataset"]["taxonomy_data"]
    )

ocean_mask_file = config["mapping"]["mask"]
mask = np.load(ocean_mask_file)
mask_inds = np.where(mask.reshape(-1) == 1)[0]

mask.shape, mask_inds

loading class to taxa


((1002, 2004),
 array([  70947,   70948,   70949, ..., 1666174, 1668177, 1668178]))

In [17]:
locs = coord_grid(mask.shape)
if config["mapping"]["mask_ocean"]:
    locs = locs[mask_inds, :]

locs.shape, locs[:5]

((499385, 2),
 array([[-34.957565,  83.70629 ],
        [-34.777832,  83.70629 ],
        [-34.598103,  83.70629 ],
        [-34.418373,  83.70629 ],
        [-33.87918 ,  83.70629 ]], dtype=float32))

In [18]:
encoded_locs = encoder.encode(locs)
print(encoded_locs[:5, :5])

tf.Tensor(
[[-0.5729696   0.21792871  0.8195766  -0.97596467]
 [-0.5703958   0.21792871  0.82136995 -0.97596467]
 [-0.5678165   0.21792871  0.82315516 -0.97596467]
 [-0.56523156  0.21792871  0.8249323  -0.97596467]
 [-0.5574435   0.21792871  0.83021486 -0.97596467]], shape=(5, 4), dtype=float32)


In [19]:
geo_model_feats = model.get_loc_emb(encoded_locs)
geo_model_feats[:5, :5]

<tf.Tensor: shape=(5, 5), dtype=float32, numpy=
array([[0.38552612, 1.5369426 , 0.28030795, 0.21224523, 0.        ],
       [0.38523668, 1.5376856 , 0.2798939 , 0.21204346, 0.        ],
       [0.38494277, 1.538428  , 0.27948073, 0.21183908, 0.        ],
       [0.3846444 , 1.5391694 , 0.27906844, 0.21163207, 0.        ],
       [0.3837222 , 1.5413891 , 0.27783695, 0.21099496, 0.        ]],
      dtype=float32)>

In [ ]:
geo_model_feats[:5, :5]